In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import segmentation_models_pytorch as smp

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
from torchvision.models import resnet34, ResNet34_Weights
#scSE блок — для улучшения обучения. В отличие от U-Net, не может быть внедрен в параметрах
#FCN для шести каналов и 12 классов

class scSEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

        self.conv_sse = nn.Conv2d(channels, 1, kernel_size=1, bias=False)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        cse = self.gap(x).view(x.size(0), -1)
        cse = self.fc(cse).view(x.size(0), x.size(1), 1, 1)

        sse = self.conv_sse(x)
        sse = self.sigmoid(sse)

        return x + x * cse + x * sse

class FCNResNet34(nn.Module):
    def __init__(self, num_classes=12, use_scse=True):
        super().__init__()
        self.use_scse = use_scse
        self.backbone = resnet34(weights=ResNet34_Weights.IMAGENET1K_V1)

        self.backbone = nn.Sequential(*list(self.backbone.children())[:-2])

        old_conv = self.backbone[0]
        new_conv = nn.Conv2d(
            6, 64, kernel_size=7, stride=2, padding=3, bias=False
        )
        with torch.no_grad():
            new_conv.weight[:, 0] = old_conv.weight[:, 0]  
            new_conv.weight[:, 1] = old_conv.weight[:, 1]  
            new_conv.weight[:, 2] = old_conv.weight[:, 2]  
            
            new_conv.weight[:, 3] = old_conv.weight[:, 2]  
            new_conv.weight[:, 4] = old_conv.weight[:, 1]  
            new_conv.weight[:, 5] = old_conv.weight[:, 0]  
        
        self.backbone[0] = new_conv
        
        if use_scse:
            self.classifier = nn.Sequential(
                nn.Conv2d(512, 256, kernel_size=3, padding=1),
                nn.ReLU(inplace=True),
                scSEBlock(256), 
                nn.Conv2d(256, 256, kernel_size=3, padding=1),
                nn.ReLU(inplace=True),
                scSEBlock(256), 
                nn.Conv2d(256, num_classes, kernel_size=1)
            )
            
            self.aux_classifier = nn.Sequential(
                nn.Conv2d(256, 256, kernel_size=3, padding=1),
                nn.ReLU(inplace=True),
                scSEBlock(256),  # scSE для вспомогательного выхода
                nn.Conv2d(256, num_classes, kernel_size=1)
            )
        else:
            self.classifier = nn.Sequential(
                nn.Conv2d(512, 256, kernel_size=3, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(256, 256, kernel_size=3, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(256, num_classes, kernel_size=1)
            )
            
            self.aux_classifier = nn.Sequential(
                nn.Conv2d(256, 256, kernel_size=3, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(256, num_classes, kernel_size=1)
            )
    
    def forward(self, x):
        input_shape = x.shape[-2:]
        
        features = []
        for layer in self.backbone:
            x = layer(x)
            features.append(x)
        
        out = self.classifier(features[-1])
        out = F.interpolate(out, size=input_shape, mode='bilinear', align_corners=False)
        
        aux = self.aux_classifier(features[-2])
        aux = F.interpolate(aux, size=input_shape, mode='bilinear', align_corners=False)
        
        return {'out': out, 'aux': aux}

model = FCNResNet34(num_classes=12, use_scse=True).to(device)

In [ ]:
#UNet для шести каналов и 12 классов

model = smp.Unet(encoder_name="resnet34", encoder_weights="imagenet", decoder_attention_type='scse', in_channels=6, classes=12)

old_conv = model.encoder.conv1
new_conv = nn.Conv2d(6, 64, kernel_size=7, stride=2, padding=3, bias=False)
with torch.no_grad():
    new_conv.weight[:, 0] = old_conv.weight[:, 0] 
    new_conv.weight[:, 1] = old_conv.weight[:, 1]   
    new_conv.weight[:, 2] = old_conv.weight[:, 2]  
    
    new_conv.weight[:, 3] = old_conv.weight[:, 2] 
    new_conv.weight[:, 4] = old_conv.weight[:, 1] 
    new_conv.weight[:, 5] = old_conv.weight[:, 0] 

model.encoder.conv1 = new_conv

In [ ]:
# PAN для шести каналов и 12 классов

#scSE блок — для улучшения обучения. В отличие от U-Net, не может быть внедрен в параметрах

class scSEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

        self.conv_sse = nn.Conv2d(channels, 1, kernel_size=1, bias=False)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        cse = self.gap(x).view(x.size(0), -1)
        cse = self.fc(cse).view(x.size(0), x.size(1), 1, 1)

        sse = self.conv_sse(x)
        sse = self.sigmoid(sse)

        return x + x * cse + x * sse

class PANWithSCSE(smp.PAN):
    def __init__(self, num_classes=12, use_scse=True):
        super().__init__(
            encoder_name='resnet34',
            encoder_weights='imagenet',
            in_channels=3,
            classes=num_classes,
            activation='softmax'
        )
        
        self.use_scse = use_scse
        
        with torch.no_grad():
            new_conv.weight[:, 0] = old_conv.weight[:, 0] 
            new_conv.weight[:, 1] = old_conv.weight[:, 1]  
            new_conv.weight[:, 2] = old_conv.weight[:, 2]  
            
            new_conv.weight[:, 3] = old_conv.weight[:, 2]  
            new_conv.weight[:, 4] = old_conv.weight[:, 1]  
            new_conv.weight[:, 5] = old_conv.weight[:, 0]
        
        self.encoder.in_channels = 6
        
        if self.use_scse:
            self.scse_final = scSEBlock(32)
    
    def forward(self, x):
        features = self.encoder(x)
        decoder_output = self.decoder(features[-1])
        
        if self.use_scse:
            decoder_output = self.scse_final(decoder_output)
        
        masks = self.segmentation_head(decoder_output)
        
        return masks
model = PAN(num_classes=12).to(device)

In [ ]:
# Функции потерь

class FocalLoss(nn.Module): # Фокальная кросс-энтропия (с учетом игнорирования классов)
    def __init__(self, alpha=None, gamma=2.0, ignore_index=255, eps=1e-7):
        super().__init__()
        self.gamma = gamma
        self.ignore_index = ignore_index
        self.eps = eps
        
        if alpha is not None:
            alpha = torch.tensor(alpha, dtype=torch.float32)
        self.register_buffer('alpha', alpha)
        
    def forward(self, pred, target):
        ce_loss = F.cross_entropy(pred, target, weight=None,
                                  ignore_index=self.ignore_index, reduction='none')
        pt = torch.exp(-ce_loss).clamp(self.eps, 1 - self.eps)
        focal_weight = (1 - pt) ** self.gamma
        
        if self.alpha is not None:
            alpha_t = self.alpha[target.clamp(0, len(self.alpha) - 1)]
            alpha_t = alpha_t * (target != self.ignore_index).float()
            focal_loss = alpha_t * focal_weight * ce_loss
        else:
            focal_loss = focal_weight * ce_loss
        
        num_valid = (target != self.ignore_index).sum()
        return focal_loss.sum() / num_valid.clamp(min=1)


class DiceLoss(nn.Module): # Функция потерь Дайса 
    def __init__(self, mode='multiclass', from_logits=True, ignore_index=255):
        super().__init__()
        self.dice = smp.losses.DiceLoss(
            mode=mode,
            from_logits=from_logits,
            ignore_index=ignore_index
        )
    
    def forward(self, pred, target):
        return self.dice(pred, target)


class CombinedLoss(nn.Module): # Комбинированная функция потерь: функция Дайса и фокальная кросс-энтропия
    def __init__(self, focal_weight=0.5, dice_weight=0.5, 
                 mode='multiclass', gamma=2.0, ignore_index=255, alpha=None, eps=1e-7, normalized=False, from_logits=True):
        super().__init__()

        self.focal = FocalLoss(alpha=alpha, gamma=gamma, ignore_index=ignore_index, eps=eps)
    
        self.dice = smp.losses.DiceLoss(
            mode=mode,
            from_logits=from_logits,
            ignore_index=ignore_index
        )
        self.focal_weight = focal_weight
        self.dice_weight = dice_weight
    
    def forward(self, pred, target):
        return (self.focal_weight * self.focal(pred, target) + 
                self.dice_weight * self.dice(pred, target))


focal_loss = FocalLoss(
    alpha=None,
    gamma=2.0,
    ignore_index=255
).to(device)


dice_loss = DiceLoss(
    mode='multiclass',
    from_logits=True,
    ignore_index=255 
).to(device)

combined_loss = CombinedLoss(
    focal_weight=0.3,
    dice_weight=0.7
).to(device)

In [ ]:
#Общая точность для вывода во время обучения

def overall_accuracy(pred, target):
    pred = pred.argmax(dim=1)
    mask = target != 255
    return (pred[mask] == target[mask]).float().mean()

oa = overall_accuracy